<a href="https://colab.research.google.com/github/nabilalidya/Probabilitas-Statistika/blob/main/TugasProbstat_Nabila3TIE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import re
import pandas as pd
import numpy as np
from scipy import stats
from google.colab import files

print("✅ Impor pustaka berhasil.")

In [ ]:
up = files.upload()
nama_file = list(up.keys())[0]
print("File terbaca:", nama_file)

In [ ]:
df = pd.read_excel(nama_file)

# Mengambil hanya kolom butir Likert berformat Dimensi [Pernyataan]
kolom_butir_likert = [c for c in df.columns if '[' in c and ']' in c]
butir = df[kolom_butir_likert].copy()

print("Jumlah responden:", len(df))
print("Jumlah butir    :", butir.shape[1])
print("Data kosong     :", int(butir.isna().sum().sum()))

In [ ]:
skala = {
    "Sangat Tidak Setuju (STS)": 1,
    "Tidak Setuju (TS)": 2,
    "Netral (N)": 3,
    "Setuju (S)": 4,
    "Sangat Setuju (SS)": 5,
}

num = butir.replace(skala).apply(pd.to_numeric, errors="coerce")
print("Sel yang gagal dikonversi (harus 0):", int(num.isna().sum().sum()))


In [ ]:
dimensi = {}
for kolom in butir.columns:
    nama = re.match(r"^(.*?) \[", kolom).group(1).strip()
    dimensi.setdefault(nama, []).append(kolom)

print("--- DAFTAR DIMENSI & JUMLAH BUTIR ---")
for nama, daftar in dimensi.items():
    print(f"• {nama}: {len(daftar)} butir")

In [ ]:
n = len(num)
t = stats.t.ppf(0.975, n - 2)
r_tabel = t / np.sqrt((n - 2) + t**2)
print(f"n = {n} | r tabel (a=0,05; df={n-2}) = {r_tabel:.3f}\n")

hasil_validitas = []
for nama, daftar in dimensi.items():
    x = num[daftar]
    total = x.sum(axis=1)
    for i, kolom in enumerate(daftar, 1):
        r, p = stats.pearsonr(x[kolom], total)
        r_koreksi, _ = stats.pearsonr(x[kolom], total - x[kolom])
        hasil_validitas.append({
            "Dimensi": nama,
            "Butir": f"B{i}",
            "r hitung": round(r, 3),
            "Corrected r": round(r_koreksi, 3),
            "p-value": round(p, 4),
            "Keterangan": "Valid" if r > r_tabel else "Tidak valid"
        })

tabel_validitas = pd.DataFrame(hasil_validitas)
display(tabel_validitas)

In [ ]:
def cronbach_alpha(x):
    k = x.shape[1]
    return k / (k - 1) * (1 - x.var(ddof=1).sum() / x.sum(axis=1).var(ddof=1))

def kategori(a):
    return ("Sangat baik" if a >= 0.90 else "Baik" if a >= 0.80 else "Reliabel" if a >= 0.70 else "Tidak reliabel")

hasil_reliabilitas = []
alpha_dihapus = []

for nama, daftar in dimensi.items():
    x = num[daftar]
    a = cronbach_alpha(x)
    hasil_reliabilitas.append({
        "Dimensi": nama,
        "Jumlah butir": len(daftar),
        "Cronbach's alpha": round(a, 3),
        "Kategori": kategori(a)
    })
    for i, kolom in enumerate(daftar, 1):
        if len(daftar) > 2:
            alpha_dihapus.append({
                "Dimensi": nama,
                "Butir": f"B{i}",
                "Alpha jika dihapus": round(cronbach_alpha(x.drop(columns=kolom)), 3)
            })

tabel_reliabilitas = pd.DataFrame(hasil_reliabilitas)
tabel_alpha_dihapus = pd.DataFrame(alpha_dihapus)

print("=== TABEL RELIABILTAS PER DIMENSI ===")
display(tabel_reliabilitas)

print("\n=== TABEL ALFA JIKA BUTIR DIHAPUS ===")
display(tabel_alpha_dihapus)

In [ ]:
# Eksperimen Latihan 3: Pengaruh Penghapusan B4 pada Dimensi Penggunaan AI
dim_penggunaan = dimensi["Penggunaan AI pada tugas atau projek kuliah"]
x_penggunaan = num[dim_penggunaan]
alpha_sebelum = cronbach_alpha(x_penggunaan)
alpha_sesudah = cronbach_alpha(x_penggunaan.drop(columns=dim_penggunaan[3]))

print(f"Latihan 3: Cronbach's Alpha sebelum B4 dihapus = {alpha_sebelum:.3f}")
print(f"Latihan 3: Cronbach's Alpha setelah B4 dihapus  = {alpha_sesudah:.3f}")

# Eksperimen Latihan 4: Pengaruh Ukuran Sampel (Subset n=20)
num_20 = num.sample(20, random_state=1)
t_20 = stats.t.ppf(0.975, 18)
r_tabel_20 = t_20 / np.sqrt(18 + t_20**2)

print(f"\nLatihan 4: Pada n=20, r tabel naik dari {r_tabel:.3f} menjadi {r_tabel_20:.3f}")


In [ ]:
topik_kuesioner = {
    "Topik 1: Penggunaan Platform AI & Dampak Psikologis": [
        "Penggunaan AI pada tugas atau projek kuliah",
        "Aspek Psikologis penggunaan AI pada tugas atau projek kuliah"
    ],
    "Topik 2: Pengaruh Kebiasaan Teknologi Digital & Prokrastinasi Akademik": [
        "Kebiasaan penggunaan teknologi digital",
        "Prokrastinasi akademik.",
        "Rsspon emosional"
    ]
}

hasil_topik = []
for nama_topik, daftar_dim dalam topik_kuesioner.items():
    cols = []
    for d in daftar_dim:
        cols.extend(dimensi[d])
    x = num[cols]
    a = cronbach_alpha(x)
    hasil_topik.append({
        "Topik Utama Kuesioner": nama_topik,
        "Jumlah Dimensi": len(daftar_dim),
        "Jumlah Butir": len(cols),
        "Cronbach's Alpha": round(a, 3),
        "Kategori": kategori(a)
    })

tabel_topik = pd.DataFrame(hasil_topik)
display(tabel_topik)

In [ ]:
gabung = tabel_validitas.merge(tabel_alpha_dihapus, on=["Dimensi", "Butir"], how="left")

with pd.ExcelWriter("hasil_uji_validitas_reliabilitas.xlsx") as w:
    gabung.to_excel(w, sheet_name="Validitas", index=False)
    tabel_reliabilitas.to_excel(w, sheet_name="Reliabilitas_Dimensi", index=False)
    tabel_topik.to_excel(w, sheet_name="Reliabilitas_Topik", index=False)

files.download("hasil_uji_validitas_reliabilitas.xlsx")
print("✅ File Excel hasil pengujian otomatis terunduh.")